# RKD Teacher Training (Colab GPU)

This notebook prepares the environment for Colab, enforces GPU usage, and runs `run.py` for teacher train/eval.

In [ ]:
# Optional: mount Google Drive for persistent checkpoints
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def run(cmd):
    print("\n>>>", " ".join(cmd))
    subprocess.run(cmd, check=True)


REPO_URL = "https://github.com/Gabomfim/MO434.git"
REPO_ROOT = Path("/content/MO434")
RKD_ROOT = REPO_ROOT / "RKD"

if not REPO_ROOT.exists():
    run(["git", "clone", REPO_URL, str(REPO_ROOT)])
else:
    run(["git", "-C", str(REPO_ROOT), "pull"])

os.chdir(RKD_ROOT)
print("Working directory:", os.getcwd())

# Colab already includes CUDA-enabled torch in GPU runtimes,
# but install/upgrade project dependencies that may be missing.
run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tqdm",
        "h5py",
        "scipy",
        "wandb",
        "kagglehub",
    ]
)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required. In Colab, go to Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

## Configure Teacher Run

In [ ]:
# Edit these values as needed
DATASET = "cub200"  # cub200 | cars196 | stanford
BASE = "resnet50"
EMBEDDING_SIZE = 512
L2NORMALIZE = "true"
SAVE_DIR = "teacher"
DATA_DIR = "../data"

# Optimization
LR = "1e-5"
BATCH = "128"
EPOCHS = "40"
ITER_PER_EPOCH = "100"
LR_DECAY_EPOCHS = ["25", "30", "35"]
LR_DECAY_GAMMA = "0.5"

# W&B
WANDB_PROJECT = "rkd-metric-learning"
WANDB_ENTITY = ""  # optional
WANDB_RUN_NAME = "teacher-resnet50-colab"
WANDB_MODE = "online"  # online | offline | disabled

In [ ]:
from pathlib import Path
import kagglehub

if DATASET == "cub200":
    path = Path(kagglehub.dataset_download("wenewone/cub2002011"))
    data_root = path.parent if path.name == "CUB_200_2011" else path
    DATA_DIR = str(data_root)
    print("Path to dataset files:", str(path))
    print("Using --data:", DATA_DIR)
else:
    print(
        f"Skipping Kaggle download because DATASET={DATASET}. Using --data: {DATA_DIR}"
    )

In [ ]:
import run as teacher_runner

teacher_train_params = {
    "mode": "train",
    "dataset": DATASET,
    "base": BASE,
    "sample": "distance",
    "loss": "l2_triplet",
    "margin": "0.2",
    "embedding_size": EMBEDDING_SIZE,
    "l2normalize": L2NORMALIZE,
    "lr": LR,
    "lr_decay_epochs": LR_DECAY_EPOCHS,
    "lr_decay_gamma": LR_DECAY_GAMMA,
    "batch": BATCH,
    "num_image_per_class": "5",
    "epochs": EPOCHS,
    "iter_per_epoch": ITER_PER_EPOCH,
    "recall": ["1"],
    "data": DATA_DIR,
    "save_dir": SAVE_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": WANDB_RUN_NAME,
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    teacher_train_params["wandb_entity"] = WANDB_ENTITY

teacher_runner.run_with_params(teacher_train_params)

In [ ]:
import run as teacher_runner

# Evaluate teacher checkpoint
teacher_eval_params = {
    "mode": "eval",
    "dataset": DATASET,
    "base": BASE,
    "embedding_size": EMBEDDING_SIZE,
    "l2normalize": L2NORMALIZE,
    "batch": BATCH,
    "recall": ["1"],
    "load": f"{SAVE_DIR}/best.pth",
    "data": DATA_DIR,
    "wandb_project": WANDB_PROJECT,
    "wandb_run_name": WANDB_RUN_NAME + "-eval",
    "wandb_mode": WANDB_MODE,
}

if WANDB_ENTITY.strip():
    teacher_eval_params["wandb_entity"] = WANDB_ENTITY

teacher_runner.run_with_params(teacher_eval_params)